# Ridge Regression — Built from Scratch

This notebook implements **Ridge Regression** entirely from scratch using only NumPy,
then evaluates it on the pre-processed student performance dataset.

Ridge adds an L2 penalty to ordinary least squares:

$$\hat{\beta} = \arg\min_\beta \; \|y - X\beta\|_2^2 + \lambda\|\beta\|_2^2$$

which has the closed-form solution:

$$\hat{\beta} = (X^TX + \lambda I)^{-1} X^T y$$

| Task | Target | Metric |
|------|--------|--------|
| Regression | `exam_score` | RMSE, MAE, R² |
| Classification | `dropout_risk` | Accuracy, F1, ROC-AUC |

Mirrors the structure of `decision_trees.ipynb` for easy side-by-side comparison.

---
## 1 · Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROCESSED_DIR = './processed'
OUT_DIR       = '../project/outputs/ridge_regression'
os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)
os.makedirs(f'{OUT_DIR}/models',  exist_ok=True)

print('Config OK. Output directory:', OUT_DIR)

---
## 2 · Load Pre-Processed Artefacts

In [ ]:
# Scaled splits — linear models require standardised features
X_train = pd.read_csv(f'{PROCESSED_DIR}/X_train_scaled.csv', index_col=0)
X_val   = pd.read_csv(f'{PROCESSED_DIR}/X_val_scaled.csv',   index_col=0)
X_test  = pd.read_csv(f'{PROCESSED_DIR}/X_test_scaled.csv',  index_col=0)

# Regression targets
y_reg_train = pd.read_csv(f'{PROCESSED_DIR}/y_reg_train.csv', index_col=0).squeeze()
y_reg_val   = pd.read_csv(f'{PROCESSED_DIR}/y_reg_val.csv',   index_col=0).squeeze()
y_reg_test  = pd.read_csv(f'{PROCESSED_DIR}/y_reg_test.csv',  index_col=0).squeeze()

# Classification targets
y_clf_train = pd.read_csv(f'{PROCESSED_DIR}/y_clf_train.csv', index_col=0).squeeze()
y_clf_val   = pd.read_csv(f'{PROCESSED_DIR}/y_clf_val.csv',   index_col=0).squeeze()
y_clf_test  = pd.read_csv(f'{PROCESSED_DIR}/y_clf_test.csv',  index_col=0).squeeze()

feature_names = pd.read_csv(f'{PROCESSED_DIR}/feature_list.csv')['feature'].tolist()

print('Loaded splits:')
for name, arr in [('Train', X_train), ('Val', X_val), ('Test', X_test)]:
    print(f'  {name:5s}  {arr.shape[0]:6d} rows  {arr.shape[1]} features')
print(f'\nRegression target  — mean={y_reg_train.mean():.2f}, std={y_reg_train.std():.2f}')
print(f'Classification target — class balance {y_clf_train.value_counts(normalize=True).round(3).to_dict()}')

---
## 3 · Ridge Regression — Built from Scratch

### 3.1 · The Maths

**Ordinary Least Squares (OLS)** minimises the residual sum of squares:
$$\mathcal{L}_{OLS} = \|y - X\beta\|_2^2$$

**Ridge** adds an L2 penalty that shrinks all coefficients toward zero:
$$\mathcal{L}_{Ridge} = \|y - X\beta\|_2^2 + \lambda\|\beta\|_2^2$$

Setting the gradient to zero and solving analytically:
$$\nabla_\beta \mathcal{L} = -2X^T(y - X\beta) + 2\lambda\beta = 0$$
$$\Rightarrow \hat{\beta} = (X^TX + \lambda I)^{-1} X^T y$$

Key properties:
- λ = 0 → plain OLS
- λ → ∞ → all coefficients collapse to 0
- The `+ λI` term guarantees the matrix is invertible even when X is rank-deficient
- All features stay in the model (Ridge shrinks but never zeros coefficients)

In [ ]:
class RidgeRegressionScratch:
    """
    Ridge Regression implemented from scratch using the closed-form solution:

        beta = (X'X + lambda * I)^{-1} X'y

    An intercept column (bias) is prepended to X internally so the
    penalty is NOT applied to the intercept term — standard practice.

    Parameters
    ----------
    lambda_ : float
        L2 regularisation strength. 0 = plain OLS.
    """

    def __init__(self, lambda_=1.0):
        self.lambda_ = lambda_
        self.beta_   = None   # shape: (n_features + 1,)  [intercept first]

    # ── Internal helpers ────────────────────────────────────────────────────
    @staticmethod
    def _add_bias(X):
        """Prepend a column of ones to X."""
        ones = np.ones((X.shape[0], 1))
        return np.hstack([ones, X])

    # ── Fit ─────────────────────────────────────────────────────────────────
    def fit(self, X, y):
        """
        Compute the closed-form Ridge solution.

        The penalty matrix has a 0 in the (0,0) position so the intercept
        is NOT regularised — identical to sklearn's convention.
        """
        X_b  = self._add_bias(np.asarray(X, dtype=float))
        y_   = np.asarray(y, dtype=float)

        n_params = X_b.shape[1]
        # Build penalty matrix: do NOT penalise the intercept (column 0)
        pen      = self.lambda_ * np.eye(n_params)
        pen[0, 0] = 0.0

        # beta = (X'X + lambda*I)^{-1} X'y
        A = X_b.T @ X_b + pen
        b = X_b.T @ y_
        self.beta_ = np.linalg.solve(A, b)   # more stable than explicit inv
        return self

    # ── Predict ─────────────────────────────────────────────────────────────
    def predict(self, X):
        X_b = self._add_bias(np.asarray(X, dtype=float))
        return X_b @ self.beta_

    # ── Convenience properties ───────────────────────────────────────────────
    @property
    def intercept_(self):
        return self.beta_[0]

    @property
    def coef_(self):
        return self.beta_[1:]


# Quick sanity check against sklearn
from sklearn.linear_model import Ridge as SklearnRidge

test_lambda = 10.0
scratch = RidgeRegressionScratch(lambda_=test_lambda).fit(X_train, y_reg_train)
sk      = SklearnRidge(alpha=test_lambda).fit(X_train, y_reg_train)

scratch_rmse = np.sqrt(mean_squared_error(y_reg_val, scratch.predict(X_val)))
sk_rmse      = np.sqrt(mean_squared_error(y_reg_val, sk.predict(X_val)))

print(f'Scratch  val RMSE: {scratch_rmse:.6f}')
print(f'Sklearn  val RMSE: {sk_rmse:.6f}')
print(f'Max coeff difference: {np.abs(scratch.coef_ - sk.coef_).max():.2e}')   # should be ~1e-8

---
## 4 · Regression Task — Predict `exam_score`

### 4.1 · λ vs RMSE Curve (Validation Set)

Mirrors the *Depth vs RMSE* curve from the decision tree notebook.

In [ ]:
lambdas = np.logspace(-3, 5, 60)   # 0.001 … 100 000

train_rmses, val_rmses = [], []

for lam in lambdas:
    m = RidgeRegressionScratch(lambda_=lam).fit(X_train, y_reg_train)
    train_rmses.append(np.sqrt(mean_squared_error(y_reg_train, m.predict(X_train))))
    val_rmses.append(np.sqrt(mean_squared_error(y_reg_val,   m.predict(X_val))))

best_lam_idx = np.argmin(val_rmses)
best_lam_reg = lambdas[best_lam_idx]

plt.figure(figsize=(10, 4))
plt.semilogx(lambdas, train_rmses, 'o-', markersize=3, label='Train RMSE')
plt.semilogx(lambdas, val_rmses,   's-', markersize=3, label='Val RMSE')
plt.axvline(best_lam_reg, color='red', linestyle='--',
            label=f'Best λ = {best_lam_reg:.4f}')
plt.xlabel('λ (log scale)')
plt.ylabel('RMSE')
plt.title('Ridge Regression — λ vs RMSE')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_lambda_vs_rmse.png', dpi=150)
plt.show()
print(f'Best λ by val RMSE: {best_lam_reg:.4f}  (val RMSE={val_rmses[best_lam_idx]:.4f})')

### 4.2 · Coefficient Shrinkage Path

Shows how each feature's coefficient changes as λ increases — the Ridge *regularisation path*.

In [ ]:
path_lambdas = np.logspace(-3, 5, 80)
coef_paths   = []

for lam in path_lambdas:
    m = RidgeRegressionScratch(lambda_=lam).fit(X_train, y_reg_train)
    coef_paths.append(m.coef_)

coef_paths = np.array(coef_paths)   # (n_lambdas, n_features)

# Highlight only the top-10 features by absolute coefficient at best lambda
best_model_path = RidgeRegressionScratch(lambda_=best_lam_reg).fit(X_train, y_reg_train)
top10_idx = np.argsort(np.abs(best_model_path.coef_))[::-1][:10]

plt.figure(figsize=(11, 5))
for i in range(len(feature_names)):
    if i in top10_idx:
        plt.semilogx(path_lambdas, coef_paths[:, i],
                     lw=1.8, label=feature_names[i])
    else:
        plt.semilogx(path_lambdas, coef_paths[:, i],
                     color='lightgray', lw=0.6, alpha=0.5)
plt.axvline(best_lam_reg, color='red', linestyle='--', lw=1.2, label=f'Best λ={best_lam_reg:.3f}')
plt.axhline(0, color='black', lw=0.8)
plt.xlabel('λ (log scale)')
plt.ylabel('Coefficient value')
plt.title('Ridge Regularisation Path — Top 10 Features')
plt.legend(fontsize=7, ncol=2, loc='upper right')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_regularisation_path.png', dpi=150)
plt.show()

### 4.3 · Fit Best Model on Train + Val, Evaluate on Test

In [ ]:
# Combine train + val for final fit (as is conventional after tuning)
X_trainval  = pd.concat([X_train, X_val])
y_reg_trainval = pd.concat([y_reg_train, y_reg_val])

best_ridge_reg = RidgeRegressionScratch(lambda_=best_lam_reg)
best_ridge_reg.fit(X_trainval, y_reg_trainval)

y_reg_pred = best_ridge_reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
mae  = mean_absolute_error(y_reg_test, y_reg_pred)
r2   = r2_score(y_reg_test, y_reg_pred)

print('─' * 35)
print(f'  Best λ   : {best_lam_reg:.4f}')
print(f'  RMSE     : {rmse:.3f}')
print(f'  MAE      : {mae:.3f}')
print(f'  R²       : {r2:.4f}')
print('─' * 35)

### 4.4 · Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
axes[0].scatter(y_reg_test, y_reg_pred, alpha=0.3, s=8, color='steelblue')
lims = [y_reg_test.min(), y_reg_test.max()]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual exam_score')
axes[0].set_ylabel('Predicted exam_score')
axes[0].set_title(f'Actual vs Predicted  (R²={r2:.3f})')
axes[0].legend()

# Residuals
residuals = y_reg_test.values - y_reg_pred
axes[1].scatter(y_reg_pred, residuals, alpha=0.3, s=8, color='coral')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Predicted exam_score')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.suptitle('Ridge Regression — Test Set Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_actual_vs_predicted.png', dpi=150)
plt.show()

### 4.5 · Feature Importances (Coefficient Magnitudes)

Because features are standardised, coefficient magnitude is a direct proxy for importance.

In [ ]:
importances_reg = pd.Series(np.abs(best_ridge_reg.coef_), index=feature_names)
top20_reg = importances_reg.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
top20_reg.plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel('|Coefficient| (standardised features)')
plt.title('Ridge Regression — Top 20 Feature Importances')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_feature_importances.png', dpi=150)
plt.show()

### 4.6 · Residual Distribution

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(0, color='red', linestyle='--', lw=1.5)
plt.xlabel('Residual (Actual − Predicted)')
plt.ylabel('Count')
plt.title('Ridge Regression — Residual Distribution')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_residual_distribution.png', dpi=150)
plt.show()
print(f'Residual mean : {residuals.mean():.4f}')
print(f'Residual std  : {residuals.std():.4f}')

### 4.7 · Save Regression Model

In [ ]:
import joblib
joblib.dump(best_ridge_reg, f'{OUT_DIR}/models/ridge_reg.joblib')
print(f'Ridge regression model saved  (λ={best_lam_reg:.4f})')

---
## 5 · Classification Task — Predict `dropout_risk`

We implement **Ridge-regularised Logistic Regression** from scratch using
**gradient descent** (no closed form exists for the logistic loss).

The loss is:
$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n \left[y_i \log(\sigma(x_i^T\beta)) + (1-y_i)\log(1-\sigma(x_i^T\beta))\right] + \frac{\lambda}{2n}\|\beta\|_2^2$$

Gradient w.r.t. β:
$$\nabla_\beta \mathcal{L} = \frac{1}{n}X^T(\hat{y} - y) + \frac{\lambda}{n}\beta$$
(intercept not penalised)

In [ ]:
class RidgeLogisticScratch:
    """
    Logistic Regression with L2 (Ridge) regularisation, trained via
    mini-batch gradient descent with early stopping.

    Parameters
    ----------
    lambda_    : L2 penalty strength
    lr         : learning rate
    epochs     : max gradient-descent iterations
    batch_size : mini-batch size (None = full batch)
    patience   : early-stopping patience (on val loss, if val data provided)
    """

    def __init__(self, lambda_=1.0, lr=0.1, epochs=500,
                 batch_size=1024, patience=20):
        self.lambda_    = lambda_
        self.lr         = lr
        self.epochs     = epochs
        self.batch_size = batch_size
        self.patience   = patience
        self.beta_      = None
        self.history_   = {'train_loss': [], 'val_loss': []}

    @staticmethod
    def _add_bias(X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    @staticmethod
    def _sigmoid(z):
        # Numerically stable sigmoid
        return np.where(z >= 0,
                        1 / (1 + np.exp(-z)),
                        np.exp(z) / (1 + np.exp(z)))

    def _bce_loss(self, X_b, y, beta):
        n   = len(y)
        p   = self._sigmoid(X_b @ beta)
        p   = np.clip(p, 1e-12, 1 - 1e-12)
        bce = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
        # Penalise all coefficients except intercept
        reg = (self.lambda_ / (2 * n)) * np.dot(beta[1:], beta[1:])
        return bce + reg

    def fit(self, X, y, X_val=None, y_val=None):
        X_b  = self._add_bias(np.asarray(X, dtype=float))
        y_   = np.asarray(y, dtype=float)
        n, p = X_b.shape

        # Initialise weights (Xavier-style)
        rng         = np.random.default_rng(42)
        self.beta_  = rng.normal(0, 1 / np.sqrt(p), size=p)

        best_val    = float('inf')
        best_beta   = self.beta_.copy()
        patience_ct = 0

        X_val_b = self._add_bias(np.asarray(X_val, dtype=float)) if X_val is not None else None
        y_val_  = np.asarray(y_val, dtype=float) if y_val is not None else None

        bs = self.batch_size if self.batch_size else n

        for epoch in range(1, self.epochs + 1):
            # Shuffle
            idx = rng.permutation(n)
            X_shuf, y_shuf = X_b[idx], y_[idx]

            # Mini-batch gradient descent
            for start in range(0, n, bs):
                Xb = X_shuf[start:start + bs]
                yb = y_shuf[start:start + bs]
                p_hat = self._sigmoid(Xb @ self.beta_)
                grad  = (Xb.T @ (p_hat - yb)) / len(yb)
                # L2 gradient (skip intercept)
                reg_grad       = (self.lambda_ / n) * self.beta_.copy()
                reg_grad[0]    = 0.0
                self.beta_    -= self.lr * (grad + reg_grad)

            train_loss = self._bce_loss(X_b, y_, self.beta_)
            self.history_['train_loss'].append(train_loss)

            if X_val_b is not None:
                val_loss = self._bce_loss(X_val_b, y_val_, self.beta_)
                self.history_['val_loss'].append(val_loss)

                if val_loss < best_val:
                    best_val   = val_loss
                    best_beta  = self.beta_.copy()
                    patience_ct = 0
                else:
                    patience_ct += 1
                    if patience_ct >= self.patience:
                        print(f'  Early stop at epoch {epoch}  (best val loss={best_val:.5f})')
                        break

            if epoch % 50 == 0:
                vl = f"{self.history_['val_loss'][-1]:.5f}" if self.history_['val_loss'] else 'N/A'
                print(f'  Epoch {epoch:4d} | train={train_loss:.5f} | val={vl}')

        if X_val_b is not None:
            self.beta_ = best_beta
        return self

    def predict_proba(self, X):
        return self._sigmoid(self._add_bias(np.asarray(X, dtype=float)) @ self.beta_)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    @property
    def intercept_(self):
        return self.beta_[0]

    @property
    def coef_(self):
        return self.beta_[1:]


print('RidgeLogisticScratch class defined.')

### 5.1 · λ vs Val AUC Curve

In [ ]:
clf_lambdas = np.logspace(-3, 4, 20)
val_aucs    = []

for lam in clf_lambdas:
    m = RidgeLogisticScratch(lambda_=lam, lr=0.1, epochs=300, patience=15)
    m.fit(X_train, y_clf_train, X_val, y_clf_val)
    val_aucs.append(roc_auc_score(y_clf_val, m.predict_proba(X_val)))
    print(f'  λ={lam:.4f}  val AUC={val_aucs[-1]:.4f}')

best_lam_clf_idx = np.argmax(val_aucs)
best_lam_clf     = clf_lambdas[best_lam_clf_idx]

plt.figure(figsize=(9, 4))
plt.semilogx(clf_lambdas, val_aucs, 's-', color='mediumseagreen')
plt.axvline(best_lam_clf, color='red', linestyle='--',
            label=f'Best λ = {best_lam_clf:.4f}')
plt.xlabel('λ (log scale)')
plt.ylabel('Val ROC-AUC')
plt.title('Ridge Logistic — λ vs Val AUC')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_lambda_vs_auc.png', dpi=150)
plt.show()
print(f'Best λ: {best_lam_clf:.4f}  (val AUC={val_aucs[best_lam_clf_idx]:.4f})')

### 5.2 · Train Best Classifier & Show Loss Curve

In [ ]:
best_ridge_clf = RidgeLogisticScratch(lambda_=best_lam_clf, lr=0.1, epochs=500, patience=20)
best_ridge_clf.fit(X_train, y_clf_train, X_val, y_clf_val)

plt.figure(figsize=(9, 4))
plt.plot(best_ridge_clf.history_['train_loss'], label='Train loss')
plt.plot(best_ridge_clf.history_['val_loss'],   label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title(f'Ridge Logistic — Training Loss Curve  (λ={best_lam_clf:.4f})')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_loss_curve.png', dpi=150)
plt.show()

### 5.3 · Evaluate on Test Set

In [ ]:
y_proba = best_ridge_clf.predict_proba(X_test)
y_pred  = best_ridge_clf.predict(X_test)

acc = accuracy_score(y_clf_test, y_pred)
f1  = f1_score(y_clf_test, y_pred)
auc = roc_auc_score(y_clf_test, y_proba)

print('─' * 35)
print(f'  Accuracy : {acc:.4f}')
print(f'  F1 Score : {f1:.4f}')
print(f'  ROC-AUC  : {auc:.4f}')
print('─' * 35)
print()
print(classification_report(y_clf_test, y_pred, target_names=['No Risk', 'Dropout Risk']))

### 5.4 · Confusion Matrix

In [ ]:
cm = confusion_matrix(y_clf_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Risk', 'Dropout Risk'],
            yticklabels=['No Risk', 'Dropout Risk'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Ridge Logistic — Confusion Matrix\n(Accuracy={acc:.3f}, F1={f1:.3f})')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_confusion_matrix.png', dpi=150)
plt.show()

### 5.5 · ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_clf_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Ridge Logistic — ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_roc_curve.png', dpi=150)
plt.show()

### 5.6 · Feature Importances (Coefficient Magnitudes)

In [ ]:
importances_clf = pd.Series(np.abs(best_ridge_clf.coef_), index=feature_names)
top20_clf = importances_clf.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
top20_clf.plot(kind='barh', color='mediumseagreen')
plt.gca().invert_yaxis()
plt.xlabel('|Coefficient| (standardised features)')
plt.title('Ridge Logistic — Top 20 Feature Importances')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_feature_importances.png', dpi=150)
plt.show()

### 5.7 · Save Classification Model

In [ ]:
joblib.dump(best_ridge_clf, f'{OUT_DIR}/models/ridge_clf.joblib')
print(f'Ridge logistic model saved  (λ={best_lam_clf:.4f})')

---
## 6 · Results Summary

In [ ]:
summary = pd.DataFrame([
    {
        'Task':        'Regression (exam_score)',
        'Best λ':      f'{best_lam_reg:.4f}',
        'Test RMSE':   f'{rmse:.3f}',
        'Test MAE':    f'{mae:.3f}',
        'Test R²':     f'{r2:.4f}',
        'Test Acc':    '—',
        'Test F1':     '—',
        'Test AUC':    '—',
    },
    {
        'Task':        'Classification (dropout_risk)',
        'Best λ':      f'{best_lam_clf:.4f}',
        'Test RMSE':   '—',
        'Test MAE':    '—',
        'Test R²':     '—',
        'Test Acc':    f'{acc:.4f}',
        'Test F1':     f'{f1:.4f}',
        'Test AUC':    f'{auc:.4f}',
    },
])
display(summary)

print('\nAll figures saved to:', f'{OUT_DIR}/figures/')
print('All models saved to: ', f'{OUT_DIR}/models/')